# Illustrative retrieval metrics for thesis Equations (2.1) and (2.2)

This notebook uses real section boundaries from the course dataset, but **manually constructed relevance grades and rankings**. It does not execute a retriever, annotate the full corpus, or report benchmark results. The five selected sections form the entire collection for this teaching example.

**Request:** Find course material explaining how POSIX thread-pool workers wait for work, receive tasks, and terminate cleanly.

Grades: 3 = direct evidence; 2 = supporting evidence; 1 = related context; 0 = irrelevant. Binary relevance for Precision/Recall uses grade >= 2. The illustrative ranking is E, D, A, C, B; the ideal graded ranking is A, B, C, D, E.

Requirements: Python 3.12+ and Matplotlib. Run all cells in order. PDF and PNG figures are saved beside this notebook. Metric definitions follow the thesis; the nDCG background reference is Järvelin and Kekäläinen (2002), DOI: 10.1145/582415.582418.

In [1]:
from math import isclose, log2
from pathlib import Path

import matplotlib.pyplot as plt

candidates = (Path.cwd(), *Path.cwd().parents, Path.cwd() / 'rag-lab-generator')
project_root = next(
    (path for path in candidates
     if (path / 'dataset').is_dir() and (path / 'diploma paper').is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError('Run from the project, notebook directory, or its workspace parent.')

output_dir = project_root / 'diploma paper' / 'formulas_figures'
units = {
    'A': ('L4_Sample_Task_3_Thread_Pool.md', '## Description', '## Stages', 3),
    'B': ('L4_Tutorial_Synchronization.md', '## Threads pool - conditional variables', '## Dice game - barrier', 3),
    'C': ('L3_Tutorial_Threads.md', '### Thread Joining', '### Excercise', 2),
    'D': ('L4_Sample_Task_1_CSV_Chunks.md', '## Task', '## Graded stages', 1),
    'E': ('L1_Sample_Task_1_Folder_Size.md', '## Description', '## Stages', 0),
}
excerpts: dict[str, str] = {}
grades: dict[str, int] = {}
for unit, (filename, start_heading, end_heading, grade) in units.items():
    lines = (project_root / 'dataset' / filename).read_text(encoding='utf-8').splitlines()
    start = lines.index(start_heading)
    end = lines.index(end_heading, start + 1)
    excerpts[unit] = '\n'.join(lines[start:end]).strip()
    assert len(excerpts[unit].splitlines()) > 1, f'Empty evidence unit: {unit}'
    grades[unit] = grade
    print(f'{unit}: grade={grade}; {filename}; {start_heading}; lines {start + 1}-{end}')

ranking = ['E', 'D', 'A', 'C', 'B']
ideal_ranking = sorted(grades, key=grades.__getitem__, reverse=True)
relevant = {unit for unit, grade in grades.items() if grade >= 2}
assert len(ranking) == len(set(ranking)) == len(units)
assert set(ranking) == set(units)
assert relevant == {'A', 'B', 'C'}


Matplotlib is building the font cache; this may take a moment.


A: grade=3; L4_Sample_Task_3_Thread_Pool.md; ## Description; lines 8-39
B: grade=3; L4_Tutorial_Synchronization.md; ## Threads pool - conditional variables; lines 151-401
C: grade=2; L3_Tutorial_Threads.md; ### Thread Joining; lines 52-69
D: grade=1; L4_Sample_Task_1_CSV_Chunks.md; ## Task; lines 6-9
E: grade=0; L1_Sample_Task_1_Folder_Size.md; ## Description; lines 6-23


In [3]:
cutoffs = list(range(1, len(ranking) + 1))
hits = [len(set(ranking[:k]) & relevant) for k in cutoffs]
precision = [count / k for count, k in zip(hits, cutoffs)]
recall = [count / len(relevant) for count in hits]
ideal_contributions = [(2 ** grades[unit] - 1) / log2(i + 2)
                       for i, unit in enumerate(ideal_ranking)]
ranked_contributions = [(2 ** grades[unit] - 1) / log2(i + 2)
                        for i, unit in enumerate(ranking)]
ideal_dcg = sum(ideal_contributions)
ranked_dcg = sum(ranked_contributions)
ndcg = ranked_dcg / ideal_dcg

assert all(isclose(actual, expected) for actual, expected in
           zip(precision, [0, 0, 1 / 3, 1 / 2, 3 / 5]))
assert all(isclose(actual, expected) for actual, expected in
           zip(recall, [0, 0, 1 / 3, 2 / 3, 1]))
assert isclose(ideal_dcg, 13.347184833073594)
assert isclose(ranked_dcg, 8.130929753571458)
assert isclose(ndcg, 0.6091868706603432)
assert isclose(recall[-1], 1.0)
assert isclose(sum(ideal_contributions) / ideal_dcg, 1.0)

print('Constructed example, not measured retrieval results:')
print('k  unit  Precision@k  Recall@k')
for k, unit, p, r in zip(cutoffs, ranking, precision, recall):
    print(f'{k}  {unit}     {p:.4f}       {r:.4f}')
print(f'Ideal DCG@5 = {ideal_dcg:.4f}; nDCG@5 = 1.0000')
print(f'Illustrative DCG@5 = {ranked_dcg:.4f}; nDCG@5 = {ndcg:.4f}')


AssertionError: 

In [ ]:
plt.rcParams.update({'font.size': 11, 'axes.titlesize': 12, 'axes.labelsize': 11})
fig, ax = plt.subplots(figsize=(7.6, 4.2), layout='constrained')
ax.plot(cutoffs, precision, marker='o', color='#2166ac', label='Precision@k')
ax.plot(cutoffs, recall, marker='s', linestyle='--', color='#b35806', label='Recall@k')
ax.set_xticks(cutoffs, [f'{k}\n{unit}' for k, unit in zip(cutoffs, ranking)])
ax.set(xlabel='Retrieval cutoff k (evidence unit at each rank)',
       ylabel='Metric value', ylim=(-0.04, 1.08),
       title='Illustrative ranking: E, D, A, C, B; relevant set: A, B, C')
ax.grid(axis='y', alpha=0.25)
ax.legend(loc='upper left')
for extension in ('pdf', 'png'):
    fig.savefig(output_dir / f'2_1_Precision_Recall.{extension}', dpi=220)
plt.show()
plt.close(fig)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), sharey=True, layout='constrained')
panels = [
    (ideal_ranking, ideal_contributions, 'Ideal ranking', ideal_dcg, 1.0),
    (ranking, ranked_contributions, 'Constructed alternative', ranked_dcg, ndcg),
]
grade_colors = {0: '#dddddd', 1: '#92c5de', 2: '#4393c3', 3: '#2166ac'}
for ax, (order, contributions, title, dcg_value, ndcg_value) in zip(axes, panels):
    bars = ax.bar(cutoffs, contributions, color=[grade_colors[grades[unit]] for unit in order])
    ax.bar_label(bars, labels=[f'{value:.2f}' for value in contributions], padding=3, fontsize=9)
    ax.set_xticks(cutoffs, [f'{i}\n{unit} (g={grades[unit]})'
                            for i, unit in zip(cutoffs, order)])
    ax.set(xlabel='Rank i and evidence unit (relevance grade g)',
           title=f'{title}\nDCG@5 = {dcg_value:.4f}; nDCG@5 = {ndcg_value:.4f}',
           ylim=(0, 8))
    ax.grid(axis='y', alpha=0.25)
    ax.set_axisbelow(True)
axes[0].set_ylabel(r'Discounted gain $(2^{g_i}-1)/\log_2(i+1)$')
fig.suptitle('Illustrative comparison: identical evidence, different ranking order')
for extension in ('pdf', 'png'):
    fig.savefig(output_dir / f'2_2_nDCG.{extension}', dpi=220)
plt.show()
plt.close(fig)

for stem in ('2_1_Precision_Recall', '2_2_nDCG'):
    for extension in ('pdf', 'png'):
        asset = output_dir / f'{stem}.{extension}'
        assert asset.stat().st_size > 1000, f'Missing or empty plot: {asset}'
        print(f'Saved {asset.name}: {asset.stat().st_size:,} bytes')
